# 00 — Input Manifesto (Configuration Hub)

**Single point of configuration for the entire XAI pipeline.**

Reads `../manifest.csv`, validates every active run, and serialises
`manifesto_config.pkl` for consumption by all downstream notebooks.

### To add a new model, sample, or dataset
1. Add a row to `manifest.csv` — **no code changes needed here.**
2. Re-run this notebook.
3. Re-run notebooks 01, 03, 04, 05, 06.

### manifest.csv column reference
| Column | Type | Description |
|---|---|---|
| `run_id` | string | Unique identifier for this run |
| `model_name` | string | Display label (e.g. XGBoost, FE_LSDV, LSTM) |
| `model_domain` | string | Business context (e.g. Sales Forecasting) |
| `model_type` | string | `Regression` or `Classification` |
| `model_family` | string | `Linear`, `Tree`, or `Neural` — drives SHAP/LIME routing |
| `dataset` | string | Sample label (DEV / OOS / OOT_01 …) |
| `file_path` | string | Path to prediction CSV (relative to this notebook) |
| `target_col` | string | Actual values column in that CSV |
| `pred_col` | string | Prediction column in that CSV (binary class or point forecast) |
| `score_col` | string | Score column used for decile ranking — probability for Classification, same as `pred_col` for Regression |
| `n_features` | integer | Feature count used for Adjusted R² |
| `group_vars` | string | Pipe-separated grouping columns (e.g. `Segment`) |
| `tolerance` | float | Hit-band fraction of actual (e.g. 0.20 = ±20%) |
| `active` | string | `Y` to include in pipeline, `N` to skip |
| `model_pkl_path` | string | Path to model pickle file (relative to this notebook) |

In [1]:
import os, pickle
import pandas as pd

NOTEBOOK_DIR  = os.path.abspath('')
MANIFEST_PATH = os.path.join(NOTEBOOK_DIR, '..', 'manifest.csv')
OUTPUT_DIR    = os.path.join(NOTEBOOK_DIR, '..', 'Outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

VALID_MODEL_TYPES    = {'Regression', 'Classification'}
VALID_MODEL_FAMILIES = {'Linear', 'Tree', 'Neural'}

print(f'Manifest : {os.path.normpath(MANIFEST_PATH)}')
print(f'Outputs  : {os.path.normpath(OUTPUT_DIR)}')

Manifest : /Users/angshubh/SalesForecasting/Modeling/codes/XAI/manifest.csv
Outputs  : /Users/angshubh/SalesForecasting/Modeling/codes/XAI/Outputs


## 1. Load Manifest

In [2]:
manifest = pd.read_csv(MANIFEST_PATH)
print(f'Manifest loaded : {len(manifest)} total rows')
manifest

Manifest loaded : 21 total rows


,run_id,model_name,model_domain,model_type,model_family,dataset,file_path,target_col,pred_col,score_col,n_features,group_vars,tolerance,active,model_pkl_path
0,FE_LSDV_DEV,FE_LSDV,Sales Forecasting,Regression,Linear,DEV,../../../outputs/FE_LSDV/pred_dev.csv,UnitSales,pred_fe,pred_fe,42,Segment,0.2,Y,../../../outputs/FE_LSDV/FE_LSDV.pkl
1,FE_LSDV_OOS,FE_LSDV,Sales Forecasting,Regression,Linear,OOS,../../../outputs/FE_LSDV/pred_oos.csv,UnitSales,pred_fe,pred_fe,42,Segment,0.2,Y,../../../outputs/FE_LSDV/FE_LSDV.pkl
2,FE_LSDV_OOT_01,FE_LSDV,Sales Forecasting,Regression,Linear,OOT_01,../../../outputs/FE_LSDV/pred_oot_01.csv,UnitSales,pred_fe,pred_fe,42,Segment,0.2,Y,../../../outputs/FE_LSDV/FE_LSDV.pkl
3,FE_LSDV_OOT_02,FE_LSDV,Sales Forecasting,Regression,Linear,OOT_02,../../../outputs/FE_LSDV/pred_oot_02.csv,UnitSales,pred_fe,pred_fe,42,Segment,0.2,Y,../../../outputs/FE_LSDV/FE_LSDV.pkl
4,FE_LSDV_OOT_03,FE_LSDV,Sales Forecasting,Regression,Linear,OOT_03,../../../outputs/FE_LSDV/pred_oot_03.csv,UnitSales,pred_fe,pred_fe,42,Segment,0.2,Y,../../../outputs/FE_LSDV/FE_LSDV.pkl
5,FE_LSDV_OOT_04,FE_LSDV,Sales Forecasting,Regression,Linear,OOT_04,../../../outputs/FE_LSDV/pred_oot_04.csv,UnitSales,pred_fe,pred_fe,42,Segment,0.2,Y,../../../outputs/FE_LSDV/FE_LSDV.pkl
6,FE_LSDV_OOT_05,FE_LSDV,Sales Forecasting,Regression,Linear,OOT_05,../../../outputs/FE_LSDV/pred_oot_05.csv,UnitSales,pred_fe,pred_fe,42,Segment,0.2,Y,../../../outputs/FE_LSDV/FE_LSDV.pkl
7,XGBoost_DEV,XGBoost,Sales Forecasting,Regression,Tree,DEV,../../../outputs/XGBoost/pred_dev.csv,UnitSales,pred_xgb,pred_xgb,39,Segment,0.2,Y,../../../outputs/XGBoost/XGBoost.pkl
8,XGBoost_OOS,XGBoost,Sales Forecasting,Regression,Tree,OOS,../../../outputs/XGBoost/pred_oos.csv,UnitSales,pred_xgb,pred_xgb,39,Segment,0.2,Y,../../../outputs/XGBoost/XGBoost.pkl
9,XGBoost_OOT_01,XGBoost,Sales Forecasting,Regression,Tree,OOT_01,../../../outputs/XGBoost/pred_oot_01.csv,UnitSales,pred_xgb,pred_xgb,39,Segment,0.2,Y,../../../outputs/XGBoost/XGBoost.pkl


## 2. Validate Active Runs

In [3]:
active   = manifest[manifest['active'].str.upper() == 'Y'].copy()
inactive = manifest[manifest['active'].str.upper() != 'Y']
print(f'Active runs   : {len(active)}')
print(f'Inactive runs : {len(inactive)}')

validation_log = []
valid_configs  = []

for _, row in active.iterrows():
    errors = []
    run_id = row['run_id']

    abs_path = os.path.normpath(os.path.join(NOTEBOOK_DIR, row['file_path']))
    abs_pkl  = os.path.normpath(os.path.join(NOTEBOOK_DIR, row['model_pkl_path']))
    score_col = str(row.get('score_col', '') or '').strip()

    # 1. Prediction file exists
    if not os.path.exists(abs_path):
        errors.append(f"file not found: {abs_path}")
    else:
        try:
            sample_df = pd.read_csv(abs_path, nrows=5)
            cols = sample_df.columns.tolist()

            if row['target_col'] not in cols:
                errors.append(f"target_col '{row['target_col']}' not in file")
            if row['pred_col'] not in cols:
                errors.append(f"pred_col '{row['pred_col']}' not in file")
            if score_col and score_col not in cols:
                errors.append(f"score_col '{score_col}' not in file")

            group_vars = [g.strip() for g in str(row['group_vars']).split('|') if g.strip()]
            for gv in group_vars:
                if gv not in cols:
                    errors.append(f"group_var '{gv}' not in file")
        except Exception as e:
            errors.append(f"could not read file: {e}")

    # 2. Model pkl exists
    if not os.path.exists(abs_pkl):
        errors.append(f"model_pkl_path not found: {abs_pkl}")

    # 3. model_type valid
    if str(row['model_type']) not in VALID_MODEL_TYPES:
        errors.append(f"model_type must be Regression or Classification, got '{row['model_type']}'")

    # 4. model_family valid
    if str(row['model_family']) not in VALID_MODEL_FAMILIES:
        errors.append(f"model_family must be Linear / Tree / Neural, got '{row['model_family']}'")

    # 5. n_features positive integer
    try:
        assert int(row['n_features']) > 0
    except:
        errors.append(f"n_features must be a positive integer, got '{row['n_features']}'")

    # 6. tolerance in (0, 1)
    try:
        assert 0 < float(row['tolerance']) < 1
    except:
        errors.append(f"tolerance must be between 0 and 1, got '{row['tolerance']}'")

    status = 'PASS' if not errors else 'FAIL'
    validation_log.append({'run_id': run_id, 'status': status,
                           'errors': '; '.join(errors) if errors else ''})

    if status == 'PASS':
        group_vars = [g.strip() for g in str(row['group_vars']).split('|') if g.strip()]
        valid_configs.append({
            'run_id'        : run_id,
            'name'          : run_id,
            'model'         : row['model_name'],
            'model_domain'  : row['model_domain'],
            'model_type'    : row['model_type'],
            'model_family'  : row['model_family'],
            'dataset'       : row['dataset'],
            'path'          : abs_path,
            'model_pkl_path': abs_pkl,
            'target_col'    : row['target_col'],
            'pred_col'      : row['pred_col'],
            'score_col'     : score_col,
            'n_features'    : int(row['n_features']),
            'tolerance'     : float(row['tolerance']),
            'group_vars'    : group_vars,
        })

val_df = pd.DataFrame(validation_log)
n_pass = (val_df['status'] == 'PASS').sum()
n_fail = (val_df['status'] == 'FAIL').sum()
print(f'\nValidation — PASS: {n_pass}  FAIL: {n_fail}')
val_df

Active runs   : 21
Inactive runs : 0

Validation — PASS: 21  FAIL: 0


,run_id,status,errors
0,FE_LSDV_DEV,PASS,
1,FE_LSDV_OOS,PASS,
2,FE_LSDV_OOT_01,PASS,
3,FE_LSDV_OOT_02,PASS,
4,FE_LSDV_OOT_03,PASS,
5,FE_LSDV_OOT_04,PASS,
6,FE_LSDV_OOT_05,PASS,
7,XGBoost_DEV,PASS,
8,XGBoost_OOS,PASS,
9,XGBoost_OOT_01,PASS,


In [4]:
failures = val_df[val_df['status'] == 'FAIL']
if len(failures):
    print('FAILED RUNS — fix manifest.csv before proceeding:\n')
    for _, r in failures.iterrows():
        print(f'  [{r.run_id}]  {r.errors}')
else:
    print('All active runs passed validation — safe to proceed.')

All active runs passed validation — safe to proceed.


## 3. Build and Export Manifesto

In [5]:
all_group_vars = sorted(set(
    gv for cfg in valid_configs for gv in cfg['group_vars']
))

manifesto = {
    'dataset_configs': valid_configs,
    'group_vars'     : all_group_vars,
}

pkl_path = os.path.join(OUTPUT_DIR, 'manifesto_config.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(manifesto, f)

print(f'manifesto_config.pkl saved')
print(f'  Runs        : {len(valid_configs)}')
print(f'  Group vars  : {all_group_vars}')
print(f'  Path        : {pkl_path}')

manifesto_config.pkl saved
  Runs        : 21
  Group vars  : ['Segment']
  Path        : /Users/angshubh/SalesForecasting/Modeling/codes/XAI/notebooks/../Outputs/manifesto_config.pkl


## 4. Summary Review

In [6]:
summary = pd.DataFrame([{
    'run_id'       : c['run_id'],
    'model'        : c['model'],
    'model_family' : c['model_family'],
    'model_type'   : c['model_type'],
    'dataset'      : c['dataset'],
    'target_col'   : c['target_col'],
    'pred_col'     : c['pred_col'],
    'score_col'    : c['score_col'],
    'n_features'   : c['n_features'],
    'group_vars'   : '|'.join(c['group_vars']),
    'tolerance'    : c['tolerance'],
} for c in valid_configs])

print(f'Manifesto ready — {len(valid_configs)} runs configured\n')
summary

Manifesto ready — 21 runs configured



,run_id,model,model_family,model_type,dataset,target_col,pred_col,score_col,n_features,group_vars,tolerance
0,FE_LSDV_DEV,FE_LSDV,Linear,Regression,DEV,UnitSales,pred_fe,pred_fe,42,Segment,0.2
1,FE_LSDV_OOS,FE_LSDV,Linear,Regression,OOS,UnitSales,pred_fe,pred_fe,42,Segment,0.2
2,FE_LSDV_OOT_01,FE_LSDV,Linear,Regression,OOT_01,UnitSales,pred_fe,pred_fe,42,Segment,0.2
3,FE_LSDV_OOT_02,FE_LSDV,Linear,Regression,OOT_02,UnitSales,pred_fe,pred_fe,42,Segment,0.2
4,FE_LSDV_OOT_03,FE_LSDV,Linear,Regression,OOT_03,UnitSales,pred_fe,pred_fe,42,Segment,0.2
5,FE_LSDV_OOT_04,FE_LSDV,Linear,Regression,OOT_04,UnitSales,pred_fe,pred_fe,42,Segment,0.2
6,FE_LSDV_OOT_05,FE_LSDV,Linear,Regression,OOT_05,UnitSales,pred_fe,pred_fe,42,Segment,0.2
7,XGBoost_DEV,XGBoost,Tree,Regression,DEV,UnitSales,pred_xgb,pred_xgb,39,Segment,0.2
8,XGBoost_OOS,XGBoost,Tree,Regression,OOS,UnitSales,pred_xgb,pred_xgb,39,Segment,0.2
9,XGBoost_OOT_01,XGBoost,Tree,Regression,OOT_01,UnitSales,pred_xgb,pred_xgb,39,Segment,0.2
